# 01 · Setup de BigQuery

Crea el dataset y las 7 tablas del modelo (ver `docs/er_diagram.png` y `docs/normalizacion.md`)
en el orden correcto según las dependencias de FK.

Requiere un `.env` en la raíz del proyecto (ver `.env.example`) con:
```
GCP_PROJECT_ID=tu-proyecto-gcp
BQ_DATASET=ecommerce_tienda
GOOGLE_APPLICATION_CREDENTIALS=ruta/a/credenciales.json   # opcional si usas ADC
```

In [ ]:
import os
from dotenv import load_dotenv
#from google.cloud import bigquery
from google.oauth2 import service_account

load_dotenv()

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET", "ecommerce_tienda")
CREDENTIALS_PATH = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

assert PROJECT_ID, "Falta GCP_PROJECT_ID en el .env"
print(f"Proyecto: {PROJECT_ID} | Dataset: {DATASET_ID}")

ModuleNotFoundError: No module named 'google.cloud'

## Conexión al cliente de BigQuery

In [ ]:
if CREDENTIALS_PATH:
    credentials = service_account.Credentials.from_service_account_file(CREDENTIALS_PATH)
    client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
else:
    # Usa Application Default Credentials (gcloud auth application-default login)
    client = bigquery.Client(project=PROJECT_ID)

print(f"Cliente BigQuery conectado -> proyecto: {client.project}")

## Crear el dataset

Ubicación `EU` porque el negocio opera en varios países europeos.

In [ ]:
dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
dataset = bigquery.Dataset(dataset_ref)
dataset.location = "EU"

dataset = client.create_dataset(dataset, exists_ok=True)
print(f"Dataset listo: {dataset.dataset_id} (location={dataset.location})")

## Esquemas de las tablas

Tipos y nulabilidad según el diseño en `docs/normalizacion.md`.

In [ ]:
SCHEMAS = {
    "categories": [
        bigquery.SchemaField("category_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("category_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("description", "STRING", mode="NULLABLE"),
    ],
    "customers": [
        bigquery.SchemaField("customer_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("first_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("last_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("email", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("phone", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("country", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("city", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("acquisition_channel", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("registration_date", "DATE", mode="REQUIRED"),
        bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
    ],
    "products": [
        bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("category_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("product_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("description", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("sale_price", "NUMERIC", mode="REQUIRED"),
        bigquery.SchemaField("cost", "NUMERIC", mode="REQUIRED"),
        bigquery.SchemaField("stock_quantity", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("is_active", "BOOL", mode="REQUIRED"),
        bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
    ],
    "orders": [
        bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("customer_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("order_status", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("shipping_address", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("shipping_city", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("shipping_country", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("order_date", "TIMESTAMP", mode="REQUIRED"),
        bigquery.SchemaField("shipped_date", "TIMESTAMP", mode="NULLABLE"),
        bigquery.SchemaField("delivered_date", "TIMESTAMP", mode="NULLABLE"),
    ],
    "order_items": [
        bigquery.SchemaField("order_item_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("quantity", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("unit_price", "NUMERIC", mode="REQUIRED"),
        bigquery.SchemaField("discount_amount", "NUMERIC", mode="REQUIRED"),
    ],
    "payments": [
        bigquery.SchemaField("payment_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("payment_method", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("payment_status", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("amount", "NUMERIC", mode="REQUIRED"),
        bigquery.SchemaField("payment_date", "TIMESTAMP", mode="REQUIRED"),
    ],
    "reviews": [
        bigquery.SchemaField("review_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("order_item_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("rating", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("comment", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("review_date", "TIMESTAMP", mode="REQUIRED"),
    ],
}

# Orden de creación: las tablas padre antes que las que tienen FK hacia ellas
TABLE_ORDER = ["categories", "customers", "products", "orders", "order_items", "payments", "reviews"]

## Crear las tablas

In [ ]:
for table_name in TABLE_ORDER:
    table_ref = dataset_ref.table(table_name)
    table = bigquery.Table(table_ref, schema=SCHEMAS[table_name])
    table = client.create_table(table, exists_ok=True)
    print(f"✔ {table.table_id:<15} {len(table.schema)} columnas")

## Verificación

In [ ]:
print("Tablas en el dataset:")
for t in client.list_tables(dataset_ref):
    full = client.get_table(t.reference)
    print(f" - {t.table_id:<15} {full.num_rows} filas")